# Smart Travel - Feature Validation

This notebook validates the engineered features before Machine Learning.

The goal is to check whether the new semantic features make sense for real travel recommendations. If these features are misleading, clustering and recommendations will learn misleading patterns.

## Validation Questions

1. Are climate labels distributed sensibly?
2. Do travel style labels look plausible?
3. Which destinations have the highest overall score?
4. Which destinations are truly balanced?
5. Are any engineered features suspicious or too dominant?
6. Which features should move forward into K-Means?

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (10, 6)

DATA_PATH = "../data/processed/destinations_feature_engineered.csv"
df = pd.read_csv(DATA_PATH)

score_columns = [
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
]

engineered_columns = [
    "warm_destination",
    "cold_destination",
    "summer_destination",
    "winter_destination",
    "rain_risk",
    "climate_category",
    "island_destination",
    "mountain_destination",
    "coastal_destination",
    "city_destination",
    "food_scene_intensity",
    "nightlife_intensity",
    "cultural_density",
    "nature_density",
    "overall_score",
    "score_balance_std",
    "balanced_destination",
    "dominant_travel_style",
]

## 1. Dataset Check

### Question

Did the feature engineering step produce a complete dataset?

### Analysis

In [ ]:
df.shape

In [ ]:
df[engineered_columns].isnull().sum()

### Insight

The engineered dataset should have no missing values in the new feature columns. Missing values here would be a blocker before clustering.

## 2. Climate Feature Validation

### Question

Are climate features distributed sensibly, or did our thresholds classify almost everything the same way?

### Analysis

In [ ]:
df["warm_destination"].value_counts()

In [ ]:
df["climate_category"].value_counts()

In [ ]:
df[[
    "destination_name",
    "winter_avg_temp",
    "summer_avg_temp",
    "summer_avg_daily_rain",
    "warm_destination",
    "summer_destination",
    "rain_risk",
    "climate_category",
]].sort_values("summer_avg_temp", ascending=False)

In [ ]:
sns.countplot(data=df, x="climate_category", order=["Cool", "Mild", "Warm"])
plt.title("Climate Category Distribution")
plt.xlabel("Climate category")
plt.ylabel("Number of destinations")
plt.show()

### Insight

A useful climate feature should separate destinations into meaningful groups. If almost every destination is Warm or Cool, the thresholds should be revisited before modeling.

## 3. Travel Style Validation

### Question

Do the dominant travel style labels look plausible?

### Analysis

In [ ]:
df["dominant_travel_style"].value_counts()

In [ ]:
df[[
    "destination_name",
    "dominant_travel_style",
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
]].sort_values(["dominant_travel_style", "destination_name"])

In [ ]:
sns.countplot(data=df, y="dominant_travel_style", order=df["dominant_travel_style"].value_counts().index)
plt.title("Dominant Travel Style Distribution")
plt.xlabel("Number of destinations")
plt.ylabel("Dominant travel style")
plt.show()

### Insight

This feature is useful only if the categories are interpretable. For example, beach destinations should generally have strong `beach_score`, while city break destinations may appear through Food & Nightlife or Culture.

## 4. Overall Score Validation

### Question

Which destinations rank highest overall, and do those rankings make sense?

### Analysis

In [ ]:
top_overall = df.sort_values("overall_score", ascending=False).head(10)
top_overall[["destination_name", "country", "overall_score", *score_columns]]

In [ ]:
sns.barplot(data=top_overall, y="destination_name", x="overall_score")
plt.title("Top 10 Destinations By Overall Score")
plt.xlabel("Overall score")
plt.ylabel("Destination")
plt.show()

### Insight

`overall_score` is useful as a broad appeal signal, but it may favor large cities because they have more POIs. It should not be the only recommendation score.

## 5. Balance Feature Validation

### Question

Which destinations are balanced across food, nightlife, culture, nature, and beach scores?

### Analysis

In [ ]:
df["balanced_destination"].value_counts()

In [ ]:
df.sort_values("score_balance_std")[[
    "destination_name",
    "balanced_destination",
    "score_balance_std",
    *score_columns,
]].head(10)

In [ ]:
df.sort_values("score_balance_std", ascending=False)[[
    "destination_name",
    "balanced_destination",
    "score_balance_std",
    *score_columns,
]].head(10)

In [ ]:
sns.histplot(data=df, x="score_balance_std", bins=8)
plt.axvline(15, color="red", linestyle="--", label="Balanced threshold")
plt.title("Distribution Of Score Balance Standard Deviation")
plt.xlabel("Score balance standard deviation")
plt.ylabel("Number of destinations")
plt.legend()
plt.show()

### Insight

Low `score_balance_std` does not always mean a destination is strong; it can also mean the destination is consistently low across all scores. For recommendations, balance should be considered together with `overall_score`.

## 6. Suspicious Feature Checks

### Question

Are any engineered features too broad, too narrow, or surprising?

### Analysis

In [ ]:
boolean_columns = [
    "warm_destination",
    "cold_destination",
    "summer_destination",
    "winter_destination",
    "rain_risk",
    "island_destination",
    "mountain_destination",
    "coastal_destination",
    "city_destination",
    "balanced_destination",
]

for column in boolean_columns:
    print(column)
    print(df[column].value_counts())
    print()

In [ ]:
df[[
    "destination_name",
    "food_scene_intensity",
    "nightlife_intensity",
    "cultural_density",
    "nature_density",
    "overall_score",
    "dominant_travel_style",
]].sort_values("overall_score", ascending=False)

### Insight

Feature validation helps catch thresholds that are too aggressive or too loose. Any feature that marks nearly every destination the same way may not help K-Means separate natural groups.

## 7. Features To Use For K-Means

### Question

Which features should be considered for the first clustering model?

### Analysis

In [ ]:
candidate_kmeans_features = [
    "food_score",
    "nightlife_score",
    "culture_score",
    "nature_score",
    "beach_score",
    "summer_avg_temp",
    "summer_avg_daily_rain",
    "overall_score",
    "score_balance_std",
]

df[["destination_name", *candidate_kmeans_features]].head()

### Insight

The first K-Means model should start with numeric, scaled features. Categorical labels such as `dominant_travel_style` and `climate_category` are useful for interpretation after clustering, but they should not be included in the first simple K-Means model.

## Final Validation Notes

- Feature engineering created interpretable traveler-facing signals.
- Climate, destination type, and travel style features should be checked before modeling.
- `overall_score` is useful, but it may favor large cities.
- `balanced_destination` should be interpreted together with `overall_score`.
- The next step is K-Means clustering using scaled numeric features.